# Single Image

In [2]:
import torch
from transformers import Qwen2_5_VLForConditionalGeneration, AutoTokenizer, AutoProcessor
from qwen_vl_utils import process_vision_info

device = "cuda:0" if torch.cuda.is_available() else "cpu"
print(device)

# default: Load the model on the available device(s)
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    "Qwen/Qwen3-VL-2B-Instruct", torch_dtype="auto", device_map="auto"
)

# We recommend enabling flash_attention_2 for better acceleration and memory saving, especially in multi-image and video scenarios.
# model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
#     "Qwen/Qwen2.5-VL-3B-Instruct",
#     torch_dtype=torch.bfloat16,
#     attn_implementation="flash_attention_2",
#     device_map="auto",
# )

# default processer
processor = AutoProcessor.from_pretrained("Qwen/Qwen3-VL-2B-Instruct")

# The default range for the number of visual tokens per image in the model is 4-16384.
# You can set min_pixels and max_pixels according to your needs, such as a token range of 256-1280, to balance performance and cost.
# min_pixels = 256*28*28
# max_pixels = 1280*28*28
# processor = AutoProcessor.from_pretrained("Qwen/Qwen2.5-VL-3B-Instruct", min_pixels=min_pixels, max_pixels=max_pixels)
url = "../Pictures_&_Videos/Testing_Pictures/Beeker.jpg" 

messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "image",
                "image":f"{url}",
            },
            {"type": "text", "text": "what is the expression on the guy in the image?"},
        ],
    }
]

# Preparation for inference
text = processor.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)
image_inputs, video_inputs = process_vision_info(messages)
inputs = processor(
    text=[text],
    images=image_inputs,
    videos=video_inputs,
    padding=True,
    return_tensors="pt",
)
inputs = inputs.to("cuda")

# Inference: Generation of the output
generated_ids = model.generate(**inputs, max_new_tokens=128)
generated_ids_trimmed = [
    out_ids[len(in_ids) :] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
]
output_text = processor.batch_decode(
    generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
)
print(output_text)

cuda:0


You are using a model of type qwen3_vl to instantiate a model of type qwen2_5_vl. This is not supported for all configurations of models and can yield errors.
Unrecognized keys in `rope_scaling` for 'rope_type'='default': {'mrope_interleaved'}
/home/yaljnadi/anaconda3/envs/HCI/lib/python3.11/site-packages/accelerate/utils/modeling.py:1566: UserWarning: Current model requires 3744 bytes of buffer for offloaded layers, which seems does not fit any GPU's remaining memory. If you are experiencing a OOM later, please consider using offload_buffers=True.
  warnings.warn(
Some weights of Qwen2_5_VLForConditionalGeneration were not initialized from the model checkpoint at Qwen/Qwen3-VL-2B-Instruct and are newly initialized: ['model.language_model.layers.0.self_attn.k_proj.bias', 'model.language_model.layers.0.self_attn.q_proj.bias', 'model.language_model.layers.0.self_attn.v_proj.bias', 'model.language_model.layers.1.self_attn.k_proj.bias', 'model.language_model.layers.1.self_attn.q_proj.bias'

RuntimeError: Expected all tensors to be on the same device, but got index is on cuda:0, different from other tensors on cpu (when checking argument in method wrapper_CUDA__index_select)

# Live Video Feed

In [1]:
import cv2
import torch
from PIL import Image
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info

# 1. Setup Device and Model
device = "cuda:0" if torch.cuda.is_available() else "cpu"
print(device)

# Load model with bfloat16 for efficiency if on GPU
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    "Qwen/Qwen2.5-VL-3B-Instruct",
    torch_dtype=torch.bfloat16,
    attn_implementation="flash_attention_2",
    device_map="auto",
)

# Load processor
# min_pixels and max_pixels help control resolution vs speed. 
# Lower max_pixels (e.g., 640*640) increases inference speed but reduces detail.
processor = AutoProcessor.from_pretrained(
    "Qwen/Qwen2.5-VL-3B-Instruct", 
    min_pixels=256*28*28, 
    max_pixels=512*28*28
)

print("Model loaded.")

# 2. Setup Webcam
cap = cv2.VideoCapture(0)  # 0 = default webcam

if not cap.isOpened():
    raise RuntimeError("Cannot open webcam.")

print("Webcam opened. Press 'q' to quit.\n")

# 3. Main Loop
while True:
    ret, frame = cap.read()
    if not ret:
        print("Failed to grab frame.")
        break

    # Show the live feed
    cv2.imshow("Live Camera Feed", frame)

    # Convert OpenCV BGR to PIL RGB
    pil_image = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))

    # Define the chat conversation
    # We pass the PIL image object directly instead of a file path
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": pil_image},
                {"type": "text", "text": "Describe what you see."},
            ],
        }
    ]

    # Prepare inputs using Qwen's specific utilities
    text = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    
    image_inputs, video_inputs = process_vision_info(messages)
    
    inputs = processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    )
    
    # Move inputs to GPU
    inputs = inputs.to(device)

    # Generate Output
    # max_new_tokens controls how long the model talks. 
    # Lower this (e.g., 50) for faster "snappier" responses.
    generated_ids = model.generate(**inputs, max_new_tokens=128)
    
    generated_ids_trimmed = [
        out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
    ]
    
    output_text = processor.batch_decode(
        generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
    )[0]

    print(output_text)

    
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

/home/yaljnadi/anaconda3/envs/HCI/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
`torch_dtype` is deprecated! Use `dtype` instead!


cuda:0


Loading checkpoint shards: 100%|██████████| 2/2 [00:00<00:00,  2.91it/s]
The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. Note that this behavior will be extended to all models in a future release.


Model loaded.
Webcam opened. Press 'q' to quit.

The image appears to be a split-screen comparison of two similar scenes, likely taken from a video or a sequence of images. The left side shows a close-up view of a person's hand holding a black object, possibly a piece of equipment or a tool. The right side is a slightly blurred version of the same scene, suggesting that it might have been taken with a different camera or at a different time.

In the background, there are several electronic devices and screens, indicating that this scene might be taking place in a workspace or a control room. The lighting is dim, and the overall environment seems to be indoors.
The image appears to be a split-screen comparison of two similar scenes, likely taken from a video or a sequence of images. The left side shows a close-up view of a person's feet and legs, with the focus on the lower part of their body. The right side is a similar close-up but slightly different in perspective, showing more of th

# Video Feed

In [ ]:
import torch
import cv2
from PIL import Image
from transformers import AutoProcessor, AutoModelForCausalLM


device = "cuda:0" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

print("Using device:", device)


model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-VL-3B-Instruct",
    torch_dtype=torch_dtype,
    trust_remote_code=True,
    attn_implementation="eager"
).to(device)

processor = AutoProcessor.from_pretrained(
    "Qwen/Qwen2.5-VL-3B-Instruct",
    trust_remote_code=True
)


cap = cv2.VideoCapture("../Pictures_&_Videos/ROS_BAGS/front_stereo_camera/right/image_compressed/temp_stream.mp4")  # 0 = default webcam

if not cap.isOpened():
    raise RuntimeError("Cannot open webcam.")

print(" Webcam opened. Press 'q' to quit.\n")

cap.set(cv2.CAP_PROP_POS_FRAMES, 100)

prompt = ""

process_every_n_frames = 5


while True:
    ret, frame = cap.read()
    ret, frame = cap.read()
    ret, frame = cap.read()
    ret, frame = cap.read()
    ret, frame = cap.read()
    if not ret:
        print("Failed to grab frame.")
        break

    pil_image = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    pil_image = pil_image.resize((640, 480))
    
    frame = cv2.resize(frame, (640, 480))
    cv2.imshow("Live Camera Feed", frame)

    inputs = processor(
        text=prompt,
        images=pil_image,
        return_tensors="pt"
    ).to(device, torch_dtype)

    
    generated_ids = model.generate(
        input_ids=inputs["input_ids"],
        pixel_values=inputs["pixel_values"],
        max_new_tokens=512,
        num_beams=3,
        do_sample=False,
        use_cache=False
    )

    
    generated_text = processor.batch_decode(
        generated_ids, 
        skip_special_tokens=False
    )[0]

    parsed_answer = processor.post_process_generation(
        generated_text,
        task="<OD>",   
        image_size=(pil_image.width, pil_image.height)
    )

    print("\n--- NEW FRAME ---")
    print(parsed_answer['<OD>']['labels'])

    # Quit with q
    if cv2.waitKey(30) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()


Using device: cuda:0
 Webcam opened. Press 'q' to quit.


--- NEW FRAME ---
['box']

--- NEW FRAME ---
['box']

--- NEW FRAME ---
['crate']

--- NEW FRAME ---
['crate']

--- NEW FRAME ---
['crate']

--- NEW FRAME ---
['box']

--- NEW FRAME ---
['purple shipping containers in a warehouse']

--- NEW FRAME ---
['purple shipping containers in a warehouse']

--- NEW FRAME ---
['purple shipping containers in warehouse']

--- NEW FRAME ---
['a warehouse']

--- NEW FRAME ---
['box']

--- NEW FRAME ---
['a warehouse']

--- NEW FRAME ---
['warehouse']

--- NEW FRAME ---
['warehouse']

--- NEW FRAME ---
['warehouse']

--- NEW FRAME ---
['warehouse']

--- NEW FRAME ---
['warehouse']

--- NEW FRAME ---
['warehouse']

--- NEW FRAME ---
['box', 'box', 'box']

--- NEW FRAME ---
['box', 'box', 'box', 'box', 'box', 'box', 'box', 'box', 'box', 'box', 'box', 'box', 'box']

--- NEW FRAME ---
['box', 'box', 'box']

--- NEW FRAME ---
['box', 'box', 'box', 'box', 'box', 'box', 'box', 'box']

--- NEW FRAME ---